# American Gut Project

In [1]:
import math
import numpy as np
import pandas as pd
from metvae.model import MetVAE
import torch
import seaborn as sns
import random

import matplotlib.pyplot as plt
%matplotlib inline

## Data import

In [2]:
df_ms1 = pd.read_csv('../data/apg/apg_abundance_table.csv', index_col=0)
df_ms1.index = df_ms1.index.astype(str)
smd = pd.read_csv('../data/apg/apg_sample_metadata.csv', index_col=0)
smd.sort_index(inplace=True)
smd.index = smd.index.astype(str)
fmd_ms2 = pd.read_table('../data/apg/apg_feature_metadata.tsv')
fmd_ms2 = fmd_ms2.set_index('#Scan#')
fmd_ms2.index = fmd_ms2.index.astype(str)

print('data shape:', df_ms1.shape)
print('sample metadata shape:', smd.shape)
print('feature metadata (MS2) shape:', fmd_ms2.shape)

data shape: (13466, 509)
sample metadata shape: (488, 495)
feature metadata (MS2) shape: (736, 42)


## Data preprocessing

In [3]:
fmd_ms1 = df_ms1[['row m/z', 'row retention time']]

# Filter columns that end with "Peak area"
peak_area_cols = [col for col in df_ms1.columns if col.endswith(" Peak area")]
df_abundance = df_ms1[peak_area_cols]
# Rename the columns to remove " Peak area" suffix
new_column_names = {col: col.split('_')[0] for col in df_ms1.columns}
df_abundance = df_abundance.rename(columns=new_column_names)

print('Proportion of zeros: ', np.round((df_abundance == 0).sum().sum()/df_abundance.size*100), '%')

Proportion of zeros:  17.0 %


In [4]:
# Subset to samples with metadata
samp_ids = set(smd.index).intersection(df_abundance.columns)
df_abundance = df_abundance[list(samp_ids)]
df_abundance = df_abundance.loc[:, ~df_abundance.columns.duplicated()]
smd = smd[smd.index.isin(list(samp_ids))]
smd['COLLECTION_YEAR'] = smd['COLLECTION_DATE'].str.split('/').str[-1]

smd.index = ["S" + str(idx) for idx in smd.index]
df_abundance.columns = ["S" + str(col) for col in df_abundance.columns]

print('data shape:', df_abundance.shape)
print('sample metadata shape:', smd.shape)
print('feature metadata (MS2) shape:', fmd_ms2.shape)

data shape: (13466, 488)
sample metadata shape: (488, 496)
feature metadata (MS2) shape: (736, 42)


In [5]:
# Antibiotics of interest
# Oxytetracycline: 8903
# Azithromycin: 5265
# Ciprofloxacin (Cipro): 1524
# RIFAMPICIN: 1372
# Tetracycline: 2164

abx_idx = list(['8903', '5265', '1524', '1372', '2164'])
sum(df_abundance.index.isin(abx_idx))

5

In [6]:
# Manual data batching (3,000 features per batch) to optimize memory usage and accelerate model training

# First separate features of interest
features_to_keep = df_abundance.loc[abx_idx].copy()
features_to_filter = df_abundance.drop(abx_idx, axis=0)

# Define batch size
batch_size = 3000

# Calculate number of batches needed
num_features_to_filter = features_to_filter.shape[0]
num_batches = math.ceil(num_features_to_filter / batch_size)

# Create batches
for batch_idx in range(num_batches):
    # Calculate start and end indices for this batch
    start_idx = batch_idx * batch_size
    end_idx = min((batch_idx + 1) * batch_size, num_features_to_filter)
    
    # Get current batch of features
    batch_indices = features_to_filter.index[start_idx:end_idx]
    current_batch = features_to_filter.loc[batch_indices]
    
    # Combine with features_to_keep
    batch_df = pd.concat([features_to_keep, current_batch], axis=0)
    
    # Create new dataframe with appropriate name
    batch_name = f"df_abundance_batch{batch_idx + 1}"
    # This creates a variable with the batch name
    globals()[batch_name] = batch_df
    
    print(f"Created {batch_name} with shape {batch_df.shape}")
    
# Verify the batches
for batch_idx in range(num_batches):
    batch_name = f"df_abundance_batch{batch_idx + 1}"
    if batch_name in globals():
        print(f"{batch_name} shape: {globals()[batch_name].shape}")

Created df_abundance_batch1 with shape (3005, 488)
Created df_abundance_batch2 with shape (3005, 488)
Created df_abundance_batch3 with shape (3005, 488)
Created df_abundance_batch4 with shape (3005, 488)
Created df_abundance_batch5 with shape (1466, 488)
df_abundance_batch1 shape: (3005, 488)
df_abundance_batch2 shape: (3005, 488)
df_abundance_batch3 shape: (3005, 488)
df_abundance_batch4 shape: (3005, 488)
df_abundance_batch5 shape: (1466, 488)


## Run MetVAE

In [7]:
# max_epochs = 2000
# learning_rate = 1e-3
# n, d = df_abundance.shape
# latent_dim = min(n, d)

# # Store batches in a list
# batches = [
#     df_abundance_batch1,
#     df_abundance_batch2,
#     df_abundance_batch3,
#     df_abundance_batch4,
#     df_abundance_batch5
# ]

# for i, batch_df in enumerate(batches, start=1):
    
#     print(f"Processing batch {i}...")
    
#     model = MetVAE(
#         data=batch_df,
#         features_as_rows=True,
#         meta=smd,
#         categorical_covariate_keys=['COLLECTION_YEAR'],
#         latent_dim=latent_dim,
#         use_gpu=False
#     )

#     model.train(
#         batch_size=100,
#         num_workers=0,
#         max_epochs=max_epochs,
#         learning_rate=learning_rate,
#         log_every_n_steps=1
#     )

#     model.get_corr(num_sim=100, workers=-1, seed=0)
#     results_metvae = model.sparse_by_p(p_adj_method='fdr_bh', cutoff=0.01)
#     est_cor = results_metvae['sparse_estimate'].values

#     df_cor = pd.DataFrame(
#         est_cor,
#         index=batch_df.index,
#         columns=batch_df.index
#     )

#     all_idx = df_cor.columns.tolist()
#     new_order = abx_idx + [x for x in all_idx if x not in abx_idx]
#     df_cor_sort = df_cor.loc[new_order, new_order]

#     # Export
#     out_base = "../results/intermediate_results/"
#     output_name = f"{out_base}apg_corr_batch{i}.csv"
#     df_cor_sort.to_csv(output_name)
    
#     print(f"Saved {output_name}")

## Outputs

In [8]:
# Read data
batch1 = pd.read_csv("../results/intermediate_results/apg_corr_batch1.csv")
batch2 = pd.read_csv("../results/intermediate_results/apg_corr_batch2.csv")
batch3 = pd.read_csv("../results/intermediate_results/apg_corr_batch3.csv")
batch4 = pd.read_csv("../results/intermediate_results/apg_corr_batch4.csv")
batch5 = pd.read_csv("../results/intermediate_results/apg_corr_batch5.csv")

In [9]:
# Combine batches
target_cols = ["row ID", "8903", "5265", "1524", "1372", "2164"]
exclude_ids = ["8903", "5265", "1524", "1372", "2164"]

batches = [batch1, batch2, batch3, batch4, batch5]

df_abx = pd.concat([
    b[target_cols][~b["row ID"].astype(str).isin(exclude_ids)]
    for b in batches
], ignore_index=True)

In [10]:
# Function to process each metabolite
def process_feature(feature_id):
    df = df_abx[["row ID", feature_id]].copy()
    
    # Apply threshold (abs >= 0.5)
    df[feature_id] = df[feature_id].where(df[feature_id].abs() >= 0.5, 0)
    
    # Remove zeros
    df = df[df[feature_id] != 0]

    # moves index into a column named "row ID"
    df_ms1_reset = df_ms1.reset_index()

    # Forcing same ID type
    df["row ID"] = df["row ID"].astype(int)
    df_ms1_reset["row ID"] = df_ms1_reset["row ID"].astype(int)
    
    # Join with ms1 metadata
    df = df.merge(
        df_ms1_reset[["row ID", "row m/z", "row retention time"]],
        on="row ID",
        how="left"
    )
    
    # Sort descending
    df = df.sort_values(by=feature_id, ascending=False)
    
    return df

In [11]:
# Generate result dataframes
df_8903 = process_feature("8903")
df_5265 = process_feature("5265")
df_1524 = process_feature("1524")
df_1372 = process_feature("1372")
df_2164 = process_feature("2164")

# Write to Excel
output_path = "../results/outputs/apg_abx_correlations.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df_8903.to_excel(writer, sheet_name="8903", index=False)
    df_5265.to_excel(writer, sheet_name="5265", index=False)
    df_1524.to_excel(writer, sheet_name="1524", index=False)
    df_1372.to_excel(writer, sheet_name="1372", index=False)
    df_2164.to_excel(writer, sheet_name="2164", index=False)